# Matched run v3: background job, token never printed
Do NOT use Run all. Run the cells one at a time, in order. The job runs in the background on Colab's machine and writes a log to Drive, so a dropped browser does not stop it.
Secret needed: `GITHUB_TOKEN_DISTILL` (create a NEW token; revoke any old one). GPU: T4, L4 or A100. When finished: Runtime > Manage sessions > Terminate.

## 1. Setup

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, subprocess
TOKEN = userdata.get('GITHUB_TOKEN_DISTILL')
REPO = 'dedeepyakm/reasoning-trace-distillation'

def git_auth(args):
    r = subprocess.run(args, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(TOKEN, '***')
    if r.returncode != 0:
        raise RuntimeError('git failed: ' + out[-500:])
    return out

if not os.path.exists('/content/repo'):
    git_auth(['git','clone',f'https://{TOKEN}@github.com/{REPO}.git','/content/repo'])
    git_auth(['git','-C','/content/repo','remote','set-url','origin',f'https://github.com/{REPO}.git'])
else:
    git_auth(['git','-C','/content/repo','pull',f'https://{TOKEN}@github.com/{REPO}.git','main'])
DRIVE_DATA = '/content/drive/MyDrive/reasoning-distillation/data'
os.makedirs(DRIVE_DATA, exist_ok=True)
if not os.path.exists('/content/repo/data'):
    os.symlink(DRIVE_DATA, '/content/repo/data')
os.chdir('/content/repo')
print(subprocess.run(['git','log','--oneline','-3'],capture_output=True,text=True).stdout)

In [ ]:
!pip -q install transformers accelerate datasets matplotlib bitsandbytes
!nvidia-smi

In [ ]:
import sys, json
sys.path.insert(0, '/content/repo')
from common.run_logging import run_step
PY = 'python'
OUT = 'data/traces_reasoning_matched_k5.jsonl'
LOG = '/content/drive/MyDrive/reasoning-distillation/matched_run.log'
COUNT_CMD = "ps aux | grep '[t]eacher_trace_generation_reasoning_matched' | wc -l"
def n_running():
    return int(subprocess.run(COUNT_CMD, shell=True, capture_output=True, text=True).stdout.strip() or 0)
def progress():
    rows = [json.loads(l) for l in open(OUT)] if os.path.exists(OUT) else []
    probs = {r['id'] for r in rows}
    return len(rows), len(probs)

## 2. Start the run in the background (safe to press twice)

In [ ]:
if n_running():
    print('Already running. Not starting another. Use the status cell.')
else:
    cmd = ['nohup', 'python', '-u', 'generation/teacher_trace_generation_reasoning_matched.py',
           '--problems-from', 'data/cut_instruct_final.jsonl', '--out', OUT,
           '--k', '5', '--max-new-tokens', '16384', '--precision', '4bit']
    proc = subprocess.Popen(cmd, stdout=open(LOG, 'a'), stderr=subprocess.STDOUT,
                            cwd='/content/repo', start_new_session=True)
    print('Started, pid', proc.pid, '\nLog on Drive:', LOG)
    print('Traces / problems already saved before this start:', progress())

## 3. Status (run any time; also right after a reconnect)

In [ ]:
print('process running:', 'YES' if n_running() else 'NO')
t, p = progress()
print(f'traces saved: {t} (target 250)   problems with traces: {p} (target 50)')
print(subprocess.run('echo "--- last log lines ---"; tail -n 8 ' + LOG + '; echo; nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv', shell=True, capture_output=True, text=True).stdout)

## 4. After it finishes (process running: NO, 250 traces)
Run these only when section 3 says NO and 250 traces. They write results into the logs.

In [ ]:
t, p = progress()
assert not n_running() and t >= 250, f'Not finished yet: running={n_running()}, traces={t}. Do not run the analysis on partial data.'
rows = [json.loads(l) for l in open(OUT)]
n = len(rows)
hit = sum(r['hit_token_limit'] for r in rows)
noans = sum(r['predicted_answer'] is None for r in rows)
print(f'traces: {n}  problems: {len({r["id"] for r in rows})}')
print(f'hit token limit: {hit} ({100*hit/n:.1f}%)   no extractable answer: {noans} ({100*noans/n:.1f}%)')
print('max tokens used:', max(r['n_generated_tokens'] for r in rows))

In [ ]:
run_step('analyze_matched', [PY,'analysis/analyze_traces_v3.py','--trace-file',OUT,'--out','data/cut_reasoning_matched.jsonl','--with-difficulty'])

In [ ]:
run_step('compare_matched', [PY,'diagnostics/compare_problem_sets.py','--a','data/cut_instruct_final.jsonl','--b','data/cut_reasoning_matched.jsonl'])

In [ ]:
run_step('figures_matched', [PY,'reporting/make_figures.py','--cut-file','data/cut_reasoning_matched.jsonl','--label','reasoning_matched','--outdir','data/figures'])
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('data/figures/reasoning_matched_*.png')):
    print(p); display(Image(p))

## 5. Stop the run (abort). Set CONFIRM_STOP = True to really stop it.

In [ ]:
CONFIRM_STOP = False
if CONFIRM_STOP:
    subprocess.run("pkill -f '[t]eacher_trace_generation_reasoning_matched'", shell=True)
    print('stop signal sent')
else:
    print('not stopped (CONFIRM_STOP is False)')

## 6. Save the logs to GitHub (logs only; the token is never printed)

In [ ]:
subprocess.run(['git','add','logs'])
print(subprocess.run(['git','status','--short'],capture_output=True,text=True).stdout)

In [ ]:
print(git_auth(['git','-c','user.name=Dedeepya Korukonda','-c','user.email=korukondadedeepya2002@gmail.com','commit','-m','Matched run logs']))
print(git_auth(['git','push',f'https://{TOKEN}@github.com/{REPO}.git','main']))